---
title: Realized quarticity
execute:
  enabled: true
---

`q.indicator.realized_quarticity` estimates integrated quarticity from one finite return window using $\frac{n}{3}\sum r_i^4$. Its fourth-power sensitivity makes large returns especially influential.

This notebook applies the scalar estimator to trailing 21-session daily log-return windows for AAPL and SPY over five years.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
returns = pd.DataFrame({symbol: pd.Series(q.indicator.log_returns(prices[symbol]), index=prices.index[1:]) for symbol in prices})
returns.tail()

,AAPL,SPY
datetime,,
2026-07-13,0.006291,-0.007686
2026-07-14,-0.007751,0.003544
2026-07-15,0.039360,0.003956
2026-07-16,0.017435,-0.005433
2026-07-17,0.001439,-0.009946


## Calculate the indicator

A single call reduces one return window to one realized-quarticity value.

In [2]:
sample = returns["AAPL"].iloc[-21:]
q.indicator.realized_quarticity(sample)

0.00017463603962251267

## Compare with SPY

Applying the estimator to each trailing window creates a dated comparison. The $10^8$ multiplier keeps the small fourth-power values readable without changing their relative shape.

In [3]:
window = 21
comparison = pd.DataFrame(
    {
        symbol: returns[symbol].rolling(window).apply(q.indicator.realized_quarticity, raw=True)
        for symbol in returns
    }
).mul(100_000_000)
comparison.columns = [f"{symbol} realized quarticity" for symbol in comparison]
fig = q.plot.col(
    comparison,
    title=f"AAPL and SPY trailing {window}-session realized quarticity",
    ylabel="Realized quarticity (×100,000,000)",
)
fig.show(renderer="notebook_connected")